# 02. Сбор городских слоёв Малой Охты

Слои по bbox v3 (вся Охта + запас): здания, дороги, зелень, полигональная вода, сервисы (POI), транспортные узлы.
Все запросы Overpass кэшируются в `data/raw/cache/` — повторные прогоны работают офлайн и не меняют цифры.
Схема атрибутов — по образцу primorsk (AGENTS.md), хранение EPSG:4326.

In [2]:
# Настройка
import json
import time
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
import shapely
from shapely.geometry import shape, LineString, Point, Polygon
from shapely.ops import nearest_points

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
RAW, PROC, CACHE = PROJECT / "data/raw", PROJECT / "data/processed", PROJECT / "data/raw/cache"
CRS_METRIC = 32636

versions = gpd.read_file(PROC / "boundary_versions.geojson")
v3 = versions.loc[versions.version == "v3", "geometry"].iloc[0]
# bbox v3 с запасом 200 м
minx, miny, maxx, maxy = v3.bounds
db = 200 / 74000  # ~200 м в градусах по широте СПб
BB = (miny - db, minx - db, maxy + db, maxx + db)  # south, west, north, east (порядок Overpass)
print("bbox запросов:", [round(v, 4) for v in BB])

OVERPASS = "https://overpass.kumi.systems/api/interpreter"

def overpass(query, cache_name, refresh=False):
    """Запрос к Overpass с кэшем в CACHE/cache_name (зеркало kumi.systems надёжнее основного)."""
    path = CACHE / cache_name
    if path.exists() and not refresh:
        return json.load(open(path, encoding="utf-8"))
    r = requests.post(OVERPASS, data={"data": query}, timeout=300,
                      headers={"User-Agent": "malaya-okhta-strategy-academic/0.1"})
    data = r.json()
    json.dump(data, open(path, "w"), ensure_ascii=False)
    return data

bb = ",".join(f"{v:.4f}" for v in BB)
print("готово")

bbox запросов: [59.9127, 30.3916, 59.9632, 30.4513]
готово


In [6]:
# Открытие выписки PBF (СЗФО, Geofabrik, скачано 02.11.2026) с bbox v3 — рецепт из pskov-blocksnet/docs/libraries/osm-offline.md
from pyrosm import OSM

west, south, east, north = minx - db, miny - db, maxx + db, maxy + db
osm = OSM(RAW / "nwfd-latest.osm.pbf", bounding_box=[west, south, east, north])
print("PBF открыт с bbox v3+200 м")

PBF открыт с bbox v3+200 м


In [7]:
# Здания через pyrosm (включая relations)
t0 = time.time()
buildings_g = osm.get_buildings()
print("зданий в bbox:", len(buildings_g), f"({time.time() - t0:.0f} с)")
# bbox режет по прямоугольнику, а не по контуру — фильтруем по границе v3
buildings_g = buildings_g[buildings_g.intersects(v3)].copy()
print("после intersects(v3):", len(buildings_g))

Reading nwfd-latest.osm.pbf: building the geometries:  35%|███▍      | 00:06<00:10/home/mfthesun/Projects/malaya-okhta-strategy/.venv/lib/python3.12/site-packages/pyrosm/engine/geoparquet.py:127: UserWarning: 1 relation(s) extend beyond the bounding box and were returned with incomplete geometry (some member ways fall outside the box). Pass complete_relations=True to OSM(...) to fetch their full member set.
  gdf = _assemble_chunk(
                                                                                  

зданий в bbox: 2780 (7 с)
после intersects(v3): 1786


In [8]:
# Остальные слои через pyrosm: дороги, зелень, вода, сервисы, транспорт
roads_g = osm.get_data_by_custom_criteria(
    custom_filter={"highway": ["motorway", "trunk", "primary", "secondary", "tertiary",
                               "unclassified", "residential", "living_street", "pedestrian",
                               "footway", "cycleway", "service", "motorway_link", "trunk_link",
                               "primary_link", "secondary_link", "tertiary_link"]},
    filter_type="keep", keep_nodes=False, keep_relations=False)
roads_g = roads_g[roads_g.intersects(v3)].copy()
print("дорог:", len(roads_g))

# зелень: landuse + leisure (parks)
green_g = osm.get_landuse()
green_g = green_g[green_g.intersects(v3)].copy() if len(green_g) else green_g
print("слоёв землепользования:", len(green_g))
natural_g = osm.get_natural()
natural_g = natural_g[natural_g.intersects(v3)].copy() if len(natural_g) else natural_g
print("слоёв natural:", len(natural_g), "(вода, лес, берега)")

# сервисы (POI)
pois_g = osm.get_pois(custom_filter={"amenity": ["school", "kindergarten", "college", "university", "library",
                                                 "clinic", "doctors", "hospital", "pharmacy", "community_centre",
                                                 "police", "fire_station", "place_of_worship"]})
pois_g = pois_g[pois_g.intersects(v3)].copy() if len(pois_g) else pois_g
print("сервисов (amenity):", len(pois_g))

# транспорт: ж/д станции и метро
rail_g = osm.get_data_by_custom_criteria(
    custom_filter={"railway": ["station", "halt"], "station": ["subway"]},
    filter_type="keep", keep_nodes=True, keep_relations=False)
rail_g = rail_g[rail_g.intersects(v3)].copy() if len(rail_g) else rail_g
print("станций:", len(rail_g))

дорог: 6106


Reading nwfd-latest.osm.pbf: building the geometries:  35%|███▍      | 00:05<00:06/home/mfthesun/Projects/malaya-okhta-strategy/.venv/lib/python3.12/site-packages/pyrosm/engine/geoparquet.py:127: UserWarning: 16 relation(s) extend beyond the bounding box and were returned with incomplete geometry (some member ways fall outside the box). Pass complete_relations=True to OSM(...) to fetch their full member set.
  gdf = _assemble_chunk(


слоёв землепользования: 714


Reading nwfd-latest.osm.pbf: building the geometries:  35%|███▍      | 00:10<00:18/home/mfthesun/Projects/malaya-okhta-strategy/.venv/lib/python3.12/site-packages/pyrosm/engine/geoparquet.py:127: UserWarning: 8 relation(s) extend beyond the bounding box and were returned with incomplete geometry (some member ways fall outside the box). Pass complete_relations=True to OSM(...) to fetch their full member set.
  gdf = _assemble_chunk(


слоёв natural: 351 (вода, лес, берега)


сервисов (amenity): 198


станций: 3


In [9]:
# Нормализация слоя buildings: схема по образцу primorsk, сохранение в processed
building = buildings_g[["id", "building", "name", "start_date", "geometry"]].copy()
building["floors"] = pd.to_numeric(buildings_g.get("building:levels"), errors="coerce")
building = building.rename(columns={"id": "osm_id"})
building["area_m2"] = building.to_crs(CRS_METRIC).geometry.area
print("зданий:", len(building), "| суммарная площадь застройки, га:", round(building.area_m2.sum() / 1e4))
print(building.building.value_counts().head(8).to_dict())

building.to_file(PROC / "building.geojson", driver="GeoJSON")
print("сохранено:", PROC / "building.geojson")

зданий: 1786 | суммарная площадь застройки, га: 178
{'yes': 573, 'apartments': 564, 'service': 159, 'industrial': 83, 'garages': 63, 'office': 57, 'roof': 45, 'kindergarten': 38}
сохранено: /home/mfthesun/Projects/malaya-okhta-strategy/data/processed/building.geojson


In [10]:
# ОКН: фильтр ЕГРОКН по версиям + ручная привязка Ниеншанца (в ЕГРОКН нет ни адреса, ни координат)
MANUAL = {  # по образцу Коломны: объекты без координат, привязка по известному месту
    "Ниеншанц": {"lon": 30.4106, "lat": 59.9461, "note": "Охтинский мыс, раскоп «Охта-1» у Красногвардейской пл."},
}

okn = pd.read_csv(RAW / "egrokn_spb_all.csv", encoding="utf-8-sig")
okn_rows = okn.dropna(subset=["lon", "lat"]).copy()
okn_rows["note"] = ""

# добавляем вручную привязанные
for name, meta in MANUAL.items():
    src = okn[okn["name"].str.startswith(name, na=False)]
    for _, r in src.iterrows():
        okn_rows = pd.concat([okn_rows, pd.DataFrame([{**r.to_dict(), "lon": meta["lon"],
                                                        "lat": meta["lat"], "note": meta["note"]}])],
                              ignore_index=True)

okn_g = gpd.GeoDataFrame(okn_rows, geometry=gpd.points_from_xy(okn_rows["lon"], okn_rows["lat"]),
                          crs="EPSG:4326")
# оставляем только объекты в v3
okn_g = okn_g[okn_g.within(versions.loc[versions.version == "v3", "geometry"].iloc[0])].copy()
okn_g.to_file(PROC / "okn.geojson", driver="GeoJSON")
print("ОКН в v3 (с ручной привязкой):", len(okn_g))
print(okn_g["cat"].value_counts().to_dict())
print("\nфедеральные:")
for _, r in okn_g[okn_g.cat == "Федерального значения"][["name", "address"]].iterrows():
    print("  ", r["name"][:70], "|", str(r["address"])[:40])

ОКН в v3 (с ручной привязкой): 60
{'Регионального значения': 43, 'Федерального значения': 17}

федеральные:
   Могила Фомина Н.П. (1864-1943), композитора, пианиста | Санкт-Петербург, Металлистов просп., 5, 
   Могила Турнера Г.И. (1858-1941), хирурга-ортопеда | Санкт-Петербург, Металлистов просп., 5, 
   Могила Поленова А.Л. (1871-1947), нейрохирурга, травматолога | Санкт-Петербург, Металлистов просп., 5, 
   Могила Малиновского В.Ф. (1765-1814), публициста, просветителя, первог | Санкт-Петербург, Металлистов просп., 5, 
   Могила Крупчатникова М.Я. (1897-1947), ученого-артиллериста, Героя Соц | Санкт-Петербург, Металлистов просп., 5, 
   Могила Деммени Е.С. (1898-1969), основателя Ленинградского театра куко | Санкт-Петербург, Металлистов просп., 5, 
   Могила Графтио Г.О. (1869-1949), гидроэнергетика | Санкт-Петербург, Металлистов просп., 5, 
   Могила Велланского (Кавунника) Д.М. (1774-1847), медика, философа | Санкт-Петербург, Металлистов просп., 5, 
   Могила Буяльского И.В. (1789

In [12]:
# Возраст домов: вырезка из «Карты возраста домов» (HOITH) по bbox через ogr2ogr.
# Исходник (1,8 ГБ) вне репозитория: ~/Workspace/Городская идентичность/HOITH_database_231118.gpkg
# Схема: r_year_int (год), r_name, r_adress, r_floors, isApartmentBuilding; CRS метрическая внутри gpkg.
import subprocess

HOITH = Path("/home/mfthesun/Workspace/Городская идентичность/HOITH_database_231118.gpkg")
hoith_cut = RAW / "hoith_okhta.gpkg"
if HOITH.exists() and not hoith_cut.exists():
    subprocess.run(["ogr2ogr", "-f", "GPKG", str(hoith_cut), str(HOITH),
                    "all_cities_181123", "-spat", str(minx), str(miny), str(maxx), str(maxy),
                    "-spat_srs", "EPSG:4326"], check=True)

if hoith_cut.exists():
    hoith = gpd.read_file(hoith_cut).to_crs(4326)
    hoith = hoith[hoith.intersects(v3)].copy()
    hoith = hoith.reset_index().rename(columns={"index": "hoith_id"})
    hoith["year"] = pd.to_numeric(hoith.get("r_year_int"), errors="coerce")
    cols = ["hoith_id", "year", "r_name", "r_adress", "r_floors", "isApartmentBuilding", "geometry"]
    hoith[[c for c in cols if c in hoith.columns]].to_file(PROC / "house_age.geojson", driver="GeoJSON")
    print("домов с возрастом в v3:", len(hoith), "| с известным годом:", int(hoith.year.notna().sum()))
    bins = pd.cut(hoith.year, [1700, 1917, 1945, 1960, 1980, 1991, 2005, 2027],
                  labels=["до 1917", "1917–1945", "1946–1960", "1961–1980", "1981–1991", "1992–2005", "2006+"],
                  right=False)
    print(bins.value_counts().sort_index().to_dict())
else:
    print("HOITH недоступен — слой возраста домов пропускается")

домов с возрастом в v3: 1596 | с известным годом: 1095
{'до 1917': 57, '1917–1945': 90, '1946–1960': 264, '1961–1980': 471, '1981–1991': 63, '1992–2005': 42, '2006+': 108}


In [14]:
# Сохранение остальных слоёв в processed + сводка сервисов и застройки по версиям
cols_road = [c for c in ["id", "highway", "name", "maxspeed", "lanes", "geometry"] if c in roads_g.columns]
roads_g[cols_road].rename(columns={"id": "osm_id"}).to_file(PROC / "roads.geojson", driver="GeoJSON")

# зелень: landuse лес/парк + natural wood/grass
keep_land = ["forest", "grass", "meadow", "recreation_ground", "cemetery", "allotments"]
land_cols = [c for c in ["id", "landuse", "name", "geometry"] if c in green_g.columns]
green_land = green_g[green_g["landuse"].isin(keep_land)][land_cols]
keep_nat = ["wood", "scrub", "grassland", "park", "water"]
nat_cols = [c for c in ["id", "natural", "water", "name", "geometry"] if c in natural_g.columns]
green_nat = natural_g[natural_g.get("natural", pd.Series(dtype=str)).isin(keep_nat)][nat_cols]
green = pd.concat([green_land, green_nat], ignore_index=True)
green.to_file(PROC / "green.geojson", driver="GeoJSON")
print("зелень:", len(green), "объектов")

# вода: полигональные natural=water + riverbank
water = natural_g[(natural_g.get("natural") == "water") | (natural_g.get("waterway") == "riverbank")]
water = water[[c for c in ["id", "natural", "water", "waterway", "name", "geometry"] if c in water.columns]]
water.to_file(PROC / "water.geojson", driver="GeoJSON")
print("вода:", len(water), "полигонов")

# сервисы и станции
pois_cols = [c for c in ["id", "amenity", "name", "geometry"] if c in pois_g.columns]
pois_g[pois_cols].rename(columns={"id": "osm_id"}).to_file(PROC / "services.geojson", driver="GeoJSON")
rail_cols = [c for c in ["id", "railway", "station", "name", "geometry"] if c in rail_g.columns]
rail_g[rail_cols].rename(columns={"id": "osm_id"}).to_file(PROC / "stations.geojson", driver="GeoJSON")
print("сервисы и станции сохранены")

# сводка сервисов и застройки по версиям границ
bld_g = gpd.read_file(PROC / "building.geojson")
for _, row in versions.iterrows():
    inside = bld_g.within(row.geometry)
    n_s = int(pois_g.within(row.geometry).sum())
    a = bld_g[inside].area_m2.sum() / 1e4
    print(f"{row['version']}: зданий {int(inside.sum())} ({a:.0f} га), сервисов {n_s}")

зелень: 642 объектов
вода: 3 полигонов
сервисы и станции сохранены
v1: зданий 1 (0 га), сервисов 0
v2: зданий 732 (74 га), сервисов 111
v3: зданий 1779 (178 га), сервисов 198
v4: зданий 1681 (168 га), сервисов 190
